# Che cosa succede senza quella radice quadrata

*Anatomia di un LLM* · capitolo 5, L'attenzione e il blocco · script `03-03`, `scripts/03_03_perche_radice_dk.py`

Nel capitolo 2 abbiamo visto che i punteggi crescono con la lunghezza dei
vettori. Qui si guarda l'effetto sulla softmax, e si scopre che il problema non
e' estetico: senza il correttivo il modello smette proprio di imparare.

Si esegue con `python3 scripts/03_03_perche_radice_dk.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/03_03_perche_radice_dk.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import math

import torch

from llm_anatomy.utils import prepara, tabella, titolo, verifica

prepara()

T, CAMPIONI = 64, 256


def entropia_media(punteggi: torch.Tensor) -> float:
    p = punteggi.softmax(dim=-1)
    return float(-(p * (p + 1e-12).log()).sum(dim=-1).mean())

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


In [3]:
titolo("Piu' numeri, punteggi piu' sparpagliati")
righe = []
for d_k in (4, 16, 64, 256, 1024):
    q = torch.randn(CAMPIONI, T, d_k)
    k = torch.randn(CAMPIONI, T, d_k)
    grezzi = q @ k.transpose(-2, -1)
    scalati = grezzi / math.sqrt(d_k)
    righe.append([
        d_k,
        f"{grezzi.std():.2f}".replace(".", ","),
        f"{math.sqrt(d_k):.2f}".replace(".", ","),
        f"{scalati.std():.3f}".replace(".", ","),
    ])
tabella(["d_k", "dev. std grezza", "sqrt(d_k)", "dev. std scalata"], righe)
print("  la deviazione grezza segue sqrt(d_k); dopo la divisione resta uno")
print("  a ogni dimensione, che e' esattamente lo scopo del fattore")


Piu' numeri, punteggi piu' sparpagliati
---------------------------------------


d_k   dev. std grezza  sqrt(d_k)  dev. std scalata
----  ---------------  ---------  ----------------
4                1,99       2,00             0,994
16               4,00       4,00             1,000
64               8,01       8,00             1,001
256             16,01      16,00             1,001
1024            32,02      32,00             1,001
  la deviazione grezza segue sqrt(d_k); dopo la divisione resta uno
  a ogni dimensione, che e' esattamente lo scopo del fattore


In [4]:
titolo("La softmax si irrigidisce")
entropia_uniforme = math.log(T)
print(f"  entropia di una distribuzione uniforme su {T} posizioni: "
      f"{entropia_uniforme:.4f} nat")
print()
righe = []
for d_k in (4, 16, 64, 256, 1024):
    q = torch.randn(CAMPIONI, T, d_k)
    k = torch.randn(CAMPIONI, T, d_k)
    grezzi = q @ k.transpose(-2, -1)
    h_senza = entropia_media(grezzi)
    h_con = entropia_media(grezzi / math.sqrt(d_k))
    massimo = float(grezzi.softmax(-1).max(dim=-1).values.mean())
    righe.append([
        d_k,
        f"{h_senza:.3f}".replace(".", ","),
        f"{h_con:.3f}".replace(".", ","),
        f"{massimo:.3f}".replace(".", ","),
    ])
tabella(["d_k", "entropia senza", "entropia con", "peso massimo senza"], righe)
print("  senza scaling, a d_k = 1024 l'attenzione mette quasi tutta la massa su")
print("  una sola posizione: e' un puntatore, non piu' una media pesata")


La softmax si irrigidisce
-------------------------
  entropia di una distribuzione uniforme su 64 posizioni: 4.1589 nat



d_k   entropia senza  entropia con  peso massimo senza
----  --------------  ------------  ------------------
4              2,804         3,699               0,277
16             1,378         3,687               0,579
64             0,579         3,685               0,792
256            0,262         3,686               0,898
1024           0,126         3,684               0,949
  senza scaling, a d_k = 1024 l'attenzione mette quasi tutta la massa su
  una sola posizione: e' un puntatore, non piu' una media pesata


In [5]:
titolo("E il modello smette di imparare")
# Confrontare la norma del gradiente rispetto a q non direbbe nulla: il fattore
# di scala moltiplica quel gradiente per costruzione. Cio' che conta e' quanto
# il softmax stesso e' ancora derivabile, cioe' la traccia del suo jacobiano,
# somma di p(1-p): vale zero esatto quando la distribuzione e' un puntatore.
righe = []
for d_k in (16, 64, 256, 1024):
    q = torch.randn(CAMPIONI, T, d_k)
    k = torch.randn(CAMPIONI, T, d_k)
    grezzi = q @ k.transpose(-2, -1)
    for nome, s in (("senza", grezzi), ("con", grezzi / math.sqrt(d_k))):
        p = s.softmax(dim=-1)
        sensibilita = float((p * (1 - p)).sum(dim=-1).mean())
        if nome == "senza":
            senza = sensibilita
        else:
            righe.append([d_k,
                          f"{senza:.4f}".replace(".", ","),
                          f"{sensibilita:.4f}".replace(".", ","),
                          f"{sensibilita / max(senza, 1e-12):.0f}x"])
tabella(["d_k", "sensibilita' senza", "sensibilita' con", "rapporto"], righe)
print("  a d_k = 1024 il softmax non scalato e' praticamente costante rispetto")
print("  ai suoi ingressi: qualunque correzione arrivi dalla perdita, i")
print("  punteggi non si spostano piu' e la testa resta congelata dov'e'")


E il modello smette di imparare
-------------------------------


d_k   sensibilita' senza  sensibilita' con  rapporto
----  ------------------  ----------------  --------
16                0,5548            0,9599        2x
64                0,2907            0,9610        3x
256               0,1465            0,9613        7x
1024              0,0744            0,9614       13x
  a d_k = 1024 il softmax non scalato e' praticamente costante rispetto
  ai suoi ingressi: qualunque correzione arrivi dalla perdita, i
  punteggi non si spostano piu' e la testa resta congelata dov'e'


In [6]:
titolo("Verifiche")
d_k = 1024
q = torch.randn(CAMPIONI, T, d_k)
k = torch.randn(CAMPIONI, T, d_k)
grezzi = q @ k.transpose(-2, -1)
verifica(abs((grezzi / math.sqrt(d_k)).std().item() - 1.0) < 0.05,
         "dopo il fattore la deviazione standard dei punteggi e' circa uno")
verifica(entropia_media(grezzi) < 0.3 * entropia_media(grezzi / math.sqrt(d_k)),
         "senza il fattore l'entropia dell'attenzione crolla sotto un terzo")


Verifiche
---------
  [verifica superata] dopo il fattore la deviazione standard dei punteggi e' circa uno
  [verifica superata] senza il fattore l'entropia dell'attenzione crolla sotto un terzo


## Da provare

prova d_k = 1024 e misura l'entropia dell'attenzione.